In [ ]:
import pandas as pd
import shutil
import os
import numpy as np
import matplotlib.pyplot as plt
import onekey_algo.custom.components as okcomp
from onekey_algo import get_param_in_cwd

plt.rcParams['figure.dpi'] = 300
model_names = ['Clinical', 'Radiomics', 'Habitat', 'Nomogram']
# 获取配置
task = get_param_in_cwd('task_column') or 'label'
bst_model = get_param_in_cwd('sel_model') or 'LR'
labelf = r'split_info/label-CV-0.csv'
group_info = get_param_in_cwd('dataset_column') or 'group'

# 读取label文件。
labels = [task]
label_data_ = pd.read_csv(labelf)
label_data_['ID'] = label_data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
label_data_ = label_data_[['ID', group_info, task]]
label_data_ = label_data_.dropna(axis=0)

ids = label_data_['ID']
print(label_data_.columns)
label_data = label_data_[['ID'] + labels]

label_data

# 训练集-汇总

In [ ]:
import pandas as pd

subset = 'train'
Clinic_results = pd.merge(pd.read_csv(f'./results/Clinical_ExtraTrees_{subset}.csv', header=0), label_data, on='ID', how='inner')
Rad_results = pd.merge(pd.read_csv(f'./results/Rad_SVM_{subset}.csv', header=0), label_data, on='ID', how='inner')
Habitat_results = pd.merge(pd.read_csv(f'./results/Habitat_Rad_XGBoost_{subset}.csv', header=0), label_data, on='ID', how='inner')

ALL_results = pd.merge(pd.merge(Clinic_results, Rad_results, on='ID', how='inner'), 
                                Habitat_results, on='ID', how='inner')
ALL_results.columns = ['ID', '-0', model_names[0], task, 
                       '-00', model_names[1], '-l', '-000', model_names[2], '-ll']
Clinic = pd.read_csv('clinic_sel.csv')

cnames = [c for c in Clinic.columns if c not in ['ID', 'group', 'label']]
Clinic = Clinic[[c for c in Clinic.columns if c not in ['label', 'group']]]
ALL_results = pd.merge(ALL_results, Clinic, on='ID', how='inner')

ALL_results = ALL_results.dropna(axis=1)
ALL_results

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from onekey_algo.custom.components import metrics

model = LogisticRegression(random_state=0, penalty='none')
# model = SVC(probability=True, random_state=0)
data_x = ALL_results[cnames + ['Habitat']]
data_y = ALL_results[task]
from sklearn.utils import shuffle
xxxx, yyyy = shuffle(data_x, data_y)
model.fit(data_x, data_y)
results = model.predict_proba(data_x)
results = pd.DataFrame(results, index=ALL_results['ID'], columns=[f'{task}-0', f'{task}-1']).reset_index()
results.to_csv(f'./results/Nomo_{subset}.csv', index=False, header=True)
pd.DataFrame([metrics.analysis_pred_binary(ALL_results[task], results[f'{task}-1'])], 
                  columns=['acc', 'auc', '95%CI', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1', 'Threshold'])

In [ ]:
data_x.columns

In [ ]:
pred_column = [f'{task}-0', f'{task}-1']
Nomo_results = pd.merge(pd.read_csv(f'./results/Nomo_{subset}.csv', header=0), label_data, on='ID', how='inner')
gt = [np.array(d) for d in [Clinic_results[labels], 
                            Rad_results[labels], Habitat_results[labels],
                            Nomo_results[labels]]]
pred_train = [np.array(d) for d in [Clinic_results[pred_column], 
                                    Rad_results[pred_column], Habitat_results[pred_column], 
                                    Nomo_results[pred_column]]]
okcomp.comp1.draw_roc(gt, pred_train, labels=model_names, title=f"Model AUC")
plt.savefig(f'img/{subset}_auc.svg')

In [ ]:
from onekey_algo.custom.components.metrics import analysis_pred_binary
metric = []
for mname, y, score in zip(model_names, gt, pred_train):
    # 计算验证集指标
    acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y, score)
    ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
    metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Train"))
pd.DataFrame(metric, index=None, columns=['Signature', 'Accuracy', 'AUC', '95% CI', 'Sensitivity', 'Specificity', 
                                          'PPV', 'NPV', 'Precision', 'Recall', 'F1','Threshold', 'Cohort'])

In [ ]:
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.comp1 import draw_matrix

delong = []
delong_columns = []
this_delong = []
plt.figure(figsize=(5, 4))
Nomo_results.columns = ['ID', '-0000', model_names[-1], '-llll']
ALL_results = pd.merge(ALL_results, Nomo_results, on='ID', how='inner')
cm = np.zeros((len(model_names), len(model_names)))
for i, mni in enumerate(model_names):
    for j, mnj in enumerate(model_names):
        if i <= j:
            cm[i][j] = np.nan
        else:
            cm[i][j] = delong_roc_test(ALL_results[task], ALL_results[mni], ALL_results[mnj])[0][0]
cm = pd.DataFrame(cm[1:, :-1], index=model_names[1:], columns=model_names[:-1])
draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
plt.title(f'Cohort {subset} Delong')
plt.savefig(f'img/all_delong_each_cohort_{subset}.svg', bbox_inches = 'tight')
plt.show()

In [ ]:
from onekey_algo.custom.components.comp1 import plot_DCA
plot_DCA([ALL_results[model_names[0]], 
          ALL_results[model_names[1]], ALL_results[model_names[2]], 
          ALL_results[model_names[3]]], 
         ALL_results[task], title=f'Model for DCA', labels=model_names, y_min=-0.15)
plt.savefig(f'img/{subset}_dca.svg')

In [ ]:
from onekey_algo.custom.components.comp1 import draw_calibration
draw_calibration(pred_scores=pred_train, n_bins=5,# smooth=True,
                 y_test=gt, model_names=model_names)
plt.savefig(f'img/{subset}_cali.svg')

In [ ]:
from onekey_algo.custom.components import stats

hosmer = []
hosmer.append([stats.hosmer_lemeshow_test(y_true, y_pred[:,1], bins=25, remap=True) 
              for fn, y_true, y_pred in zip(model_names, gt, pred_train)])
pd.DataFrame(hosmer, columns=model_names)

# 训练Cox模型

In [ ]:
# from lifelines import CoxPHFitter

# cox_data = pd.merge(ALL_results[['ID', 'Clinic_Sig', 'Rad_Sig', 'DTL_Sig']], cox_data, on='ID', how='inner')

# cph = CoxPHFitter(penalizer=0.1)
# cph.fit(cox_data[[c for c in cox_data.columns if c != 'ID']], duration_col='event_time', event_col='event')
# cph.print_summary()

# 绘制Nomogram

In [ ]:
from onekey_algo.custom.components import nomogram
import shutil

ALL_results = ALL_results.round(decimals=2)
nomogram.risk_nomogram(ALL_results, result=task, columns=list(data_x.columns), width=7000, height=3500,
                      x_range='0.01,0.25,0.5,0.75,0.99')

# 验证集-汇总

In [ ]:
import pandas as pd

subset = 'test'
Clinic_results = pd.merge(pd.read_csv(f'./results/Clinical_ExtraTrees_{subset}.csv', header=0), label_data, on='ID', how='inner')
Rad_results = pd.merge(pd.read_csv(f'./results/Rad_RandomForest_{subset}.csv', header=0), label_data, on='ID', how='inner')
Habitat_results = pd.merge(pd.read_csv(f'./results/Habitat_Rad_XGBoost_{subset}.csv', header=0), label_data, on='ID', how='inner')

ALL_results = pd.merge(pd.merge(Clinic_results, Rad_results, on='ID', how='inner'), 
                                Habitat_results, on='ID', how='inner')
ALL_results.columns = ['ID', '-0', model_names[0], task, 
                       '-00', model_names[1], '-l', '-000', model_names[2], '-ll']
Clinic = pd.read_csv('clinic_sel.csv')
Clinic = Clinic[[c for c in Clinic.columns if c not in ['label', 'group']]]
ALL_results = pd.merge(ALL_results, Clinic, on='ID', how='inner')

ALL_results = ALL_results.dropna(axis=1)
ALL_results

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from onekey_algo.custom.components import metrics

# model = LogisticRegression(random_state=0)
# model = SVC(probability=True, random_state=0)
data_x = ALL_results[data_x.columns]
data_y = ALL_results[task]
# model.fit(data_x, data_y)
results = model.predict_proba(data_x)
results = pd.DataFrame(results, index=ALL_results['ID'], columns=[f'{task}-0', f'{task}-1']).reset_index()
results.to_csv(f'./results/Nomo_{subset}.csv', index=False, header=True)
pd.DataFrame([metrics.analysis_pred_binary(ALL_results[task], results[f'{task}-1'])], 
                  columns=['acc', 'auc', '95%CI', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1', 'Threshold'])

In [ ]:
pred_column = [f'{task}-0', f'{task}-1']
Nomo_results = pd.merge(pd.read_csv(f'./results/Nomo_{subset}.csv', header=0), label_data, on='ID', how='inner')
gt = [np.array(d) for d in [Clinic_results[labels], 
                            Rad_results[labels], Habitat_results[labels],
                            Nomo_results[labels]]]
pred_train = [np.array(d) for d in [Clinic_results[pred_column], 
                                    Rad_results[pred_column], Habitat_results[pred_column], 
                                    Nomo_results[pred_column]]]
okcomp.comp1.draw_roc(gt, pred_train, labels=model_names, title=f"Model AUC")
plt.savefig(f'img/{subset}_auc.svg')

In [ ]:
from onekey_algo.custom.components.metrics import analysis_pred_binary
for mname, y, score in zip(model_names, gt, pred_train):
    # 计算验证集指标
    acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y, score)
    ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
    metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Test"))
pd.DataFrame(metric, index=None, columns=['Signature', 'Accuracy', 'AUC', '95% CI',
                                          'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1',
                                          'Threshold', 'Cohort'])

In [ ]:
# from onekey_algo.custom.components.comp1 import calc_confusion_matrix_from_prob

# mapping = get_param_in_cwd('label_mapping')
# for mn in model_names:
#     cm = okcomp.comp1.calc_confusion_matrix_from_prob(ALL_results[mn], ALL_results[labels[0]], 
#                                                       class_mapping=mapping, num_classes=2)
#     plt.figure(figsize=(5, 4))
#     okcomp.comp1.draw_matrix(cm, norm=False, annot=True, cmap='Blues', fmt='.0f')
#     plt.savefig(f'img/{mn}_model_train_cm.svg', bbox_inches = 'tight')
#     plt.show()

In [ ]:
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.comp1 import draw_matrix

delong = []
delong_columns = []
this_delong = []
plt.figure(figsize=(5, 4))
Nomo_results.columns = ['ID', '-0000', model_names[-1], '-llll']
ALL_results = pd.merge(ALL_results, Nomo_results, on='ID', how='inner')
cm = np.zeros((len(model_names), len(model_names)))
for i, mni in enumerate(model_names):
    for j, mnj in enumerate(model_names):
        if i <= j:
            cm[i][j] = np.nan
        else:
            cm[i][j] = delong_roc_test(ALL_results[task], ALL_results[mni], ALL_results[mnj])[0][0]
cm = pd.DataFrame(cm[1:, :-1], index=model_names[1:], columns=model_names[:-1])
draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
plt.title(f'Cohort {subset} Delong')
plt.savefig(f'img/all_delong_each_cohort_{subset}.svg', bbox_inches = 'tight')
plt.show()

In [ ]:
from onekey_algo.custom.components.comp1 import plot_DCA
plot_DCA([ALL_results[model_names[0]], 
          ALL_results[model_names[1]], ALL_results[model_names[2]], 
          ALL_results[model_names[3]]], 
         ALL_results[task], title=f'Model for DCA', labels=model_names, y_min=-0.15)
plt.savefig(f'img/{subset}_dca.svg')

In [ ]:
from onekey_algo.custom.components.comp1 import draw_calibration
draw_calibration(pred_scores=pred_train, n_bins=7, remap=True, EX={'n_estimators': 2, 'max_depth':2}, #smooth=True, # window_length=7,
                 y_test=gt, model_names=model_names)
plt.savefig(f'img/{subset}_cali.svg')

In [ ]:
from onekey_algo.custom.components import stats

hosmer.append([stats.hosmer_lemeshow_test(y_true, y_pred[:,1], bins=10, remap=True) 
              for fn, y_true, y_pred in zip(model_names, gt, pred_train)])
pd.concat([pd.DataFrame(hosmer, columns=model_names), pd.DataFrame(['Train', 'Test'], columns=['Cohort'])], axis=1)